# TP2 — Bordes y líneas de la cancha (informe)

**Nombre:** Matías · **Entregas:** S1 _fecha_ · S2 _fecha_ · S3 _fecha_ · S4 _fecha_

Usa `mv.filtros` (TP2), `mv.geometria` (TP1) y `mv.imagen` (TP0).
OpenCV / SciPy aparecen solo para **comparar**. Respondé en las celdas **✍️ Respuesta**.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

from mv import filtros as f
from mv import geometria as g
from mv import imagen as im
from mv.datos import cargar_frame, mostrar

img = cargar_frame()                     # RGB uint8
H, W = img.shape[:2]
gris = im.a_gris(img).astype(float)      # [0, 255]
print(img.shape)

---
# Sesión 1 — Color

## S1.a Los canales H, S, V

In [ ]:
hsv = f.rgb_a_hsv(img)
fig, ejes = plt.subplots(1, 3, figsize=(18, 4))
for ax, k, nombre, cmap in zip(ejes, range(3), ["H (tono, grados)", "S (saturación)", "V (brillo)"], ["hsv", "gray", "gray"]):
    im_ = ax.imshow(hsv[..., k], cmap=cmap); ax.set_title(nombre); ax.axis("off"); plt.colorbar(im_, ax=ax, fraction=0.03)
plt.show()

**Preguntas**
1. ¿Qué zonas tienen H "ruidoso" (cambia de píxel a píxel)? Relacionalo con S y V en esas zonas.
2. ¿Qué canal distingue mejor las líneas de cal del pasto? ¿Y las camisetas de cada equipo?

✍️ **Respuesta:**

## S1.b Máscara de pasto con sombra: RGB (TP0) contra HSV

Simulamos una sombra fuerte sobre la mitad izquierda del frame (30 % de la luz). Si tu frame real tiene sombras de verdad, repetí la comparación ahí.

In [ ]:
sombra = img.copy()
sombra[:, : W // 2] = (sombra[:, : W // 2].astype(float) * 0.3).astype(np.uint8)

m_rgb = im.mascara_pasto(sombra)          # tu regla del TP0
m_hsv = f.mascara_pasto_hsv(sombra)
mostrar(sombra, m_rgb, m_hsv, titulos=["con sombra", "máscara RGB (TP0)", "máscara HSV"], ancho=6)
for nombre, m in (("RGB", m_rgb), ("HSV", m_hsv)):
    print(f"{nombre}: pasto en la mitad con sombra {m[:, :W//2].mean():.2f} | al sol {m[:, W//2:].mean():.2f}")

**Preguntas**
3. ¿Cuál de las dos máscaras se rompe con la sombra? Explicá **con un píxel concreto** (valores RGB antes y después) por qué.
4. ¿Hay algo que la máscara HSV marque como pasto y no lo sea, o al revés? ¿Qué umbral tocarías?

✍️ **Respuesta:**

## S1.c Ecualizar un frame oscuro

In [ ]:
oscuro = np.clip(gris * 0.35 + 10, 0, 255).astype(np.uint8)
ecual = f.ecualizar_histograma(oscuro)
fig, ejes = plt.subplots(2, 2, figsize=(14, 7))
ejes[0, 0].imshow(oscuro, cmap="gray", vmin=0, vmax=255); ejes[0, 0].set_title("oscuro")
ejes[0, 1].imshow(ecual, cmap="gray", vmin=0, vmax=255); ejes[0, 1].set_title("ecualizado")
ejes[1, 0].bar(range(256), im.histograma(oscuro), width=1); ejes[1, 0].set_title("histograma oscuro")
ejes[1, 1].bar(range(256), im.histograma(ecual), width=1); ejes[1, 1].set_title("histograma ecualizado")
for ax in ejes[0]: ax.axis("off")
plt.tight_layout(); plt.show()

**Preguntas**
5. ¿Por qué el histograma ecualizado no queda perfectamente plano?
6. ¿Qué se ve peor después de ecualizar? (Mirá zonas planas: tribuna, pasto.)

✍️ **Respuesta:**

---
# Sesión 2 — Convolución

## S2.a Tu convolución contra SciPy

In [ ]:
import scipy.ndimage as ndi
k = np.ones((7, 7)) / 49
t0 = time.perf_counter(); mio = f.convolucionar(gris, k); t_mio = time.perf_counter() - t0
t0 = time.perf_counter(); ref = ndi.convolve(gris, k, mode="mirror"); t_ref = time.perf_counter() - t0
print(f"diferencia máx: {np.abs(mio - ref).max():.2e} | mío {t_mio*1000:.0f} ms | scipy {t_ref*1000:.0f} ms")

## S2.b Modos de borde

In [ ]:
k = np.ones((41, 41)) / 41**2
salidas = [f.convolucionar(gris, k, modo) for modo in ("cero", "replicar", "espejo")]
mostrar(*salidas, titulos=["cero", "replicar", "espejo"], ancho=5)

**Preguntas**
7. Describí el artefacto de cada modo en los bordes. ¿Por qué `"cero"` oscurece?

✍️ **Respuesta:**

## S2.c Correlación contra convolución: un kernel que "mueve" la imagen

In [ ]:
k = np.zeros((41, 41)); k[20, 40] = 1.0          # un 1 a la derecha del centro
corr = f.correlacionar(gris, k, "cero")
conv = f.convolucionar(gris, k, "cero")
mostrar(gris, corr, conv, titulos=["original", "correlación", "convolución"], ancho=5)

**Preguntas**
8. ¿Para qué lado se movió la imagen en cada caso, y cuántos píxeles? Explicalo con la fórmula de cada una.

✍️ **Respuesta:**

## S2.d La escala: ¿cuándo desaparece la pelota?

Elegí una caja alrededor de la pelota (en el frame sintético está cerca de (0,45·W, 0,6·H)).

In [ ]:
# TODO (frame real): ajustá la caja a donde esté la pelota en tu frame
cx, cy = int(0.45 * W), int(0.6 * H)
caja = (cx - 40, cy - 40, cx + 40, cy + 40)
rec = im.recortar(gris, caja)
sigmas = [0.5, 1, 2, 4, 8]
mostrar(rec, *[f.desenfocar_gaussiano(rec, s) for s in sigmas], titulos=["original"] + [f"σ={s}" for s in sigmas], ancho=3)

**Preguntas**
9. ¿Desde qué σ ya no distinguís la pelota? Relacionalo con su tamaño en píxeles. ¿Qué implica para un detector que quiera ver la pelota y también ignorar el ruido?

✍️ **Respuesta:**

## S2.e Separable contra 2D

In [ ]:
# TODO: con k = kernel_gaussiano(5.0) (31 coeficientes), medí el tiempo de:
#   (a) f.filtrar_separable(gris, k)
#   (b) f.convolucionar(gris, np.outer(k, k))
# y verificá que dan lo mismo. ¿La aceleración es ~31/2? ¿Por qué podría no serlo?

## S2.f Gaussiano contra mediana con "sal y pimienta"

In [ ]:
rng = np.random.default_rng(0)
ruidosa = gris.copy()
sal = rng.random(gris.shape) < 0.03
ruidosa[sal] = rng.choice([0, 255], size=sal.sum())
mostrar(ruidosa, f.desenfocar_gaussiano(ruidosa, 1.0), f.filtro_mediana(ruidosa, 3),
        titulos=["3 % sal y pimienta", "gaussiano σ=1", "mediana 3×3"], ancho=6)

**Preguntas**
10. ¿Cuál limpia mejor y cuál conserva mejor las líneas de la cancha? ¿Por qué el gaussiano "mancha"?

✍️ **Respuesta:**

---
# Sesión 3 — Gradiente

## S3.a gx, gy, magnitud y orientación

In [ ]:
suave = f.desenfocar_gaussiano(gris, 1.0)
gx, gy = f.gradiente_sobel(suave)
mag, th = f.magnitud_orientacion(gx, gy)

# orientación como color (tono = ángulo), apagada donde el gradiente es chico
ang = (np.rad2deg(th) % 360) / 360
viz = plt.cm.hsv(ang)[..., :3] * (mag / np.percentile(mag, 99)).clip(0, 1)[..., None]
mostrar(gx, gy, mag, viz, titulos=["gx", "gy", "magnitud", "orientación (color)"], ancho=5)

**Preguntas**
11. ¿Qué líneas de la cancha aparecen fuertes en gx y cuáles en gy? ¿Por qué?
12. En la imagen de orientación, los dos lados de una misma línea de cal tienen colores opuestos. ¿Por qué?

✍️ **Respuesta:**

## S3.b Un perfil a través de una línea de cal

Tomamos una columna que cruce la lateral lejana (en el frame sintético está en la fila ~0,19·H).

In [ ]:
# TODO (frame real): elegí una columna x0 y un rango de filas que crucen una línea de cal
x0 = W // 3
y_ini, y_fin = int(0.12 * H), int(0.28 * H)
filas = np.arange(y_ini, y_fin)

rng = np.random.default_rng(1)
perfil = gris[y_ini:y_fin, x0] + rng.normal(0, 4, len(filas))      # le agregamos un poco de ruido
deriv_cruda = np.gradient(perfil)
deriv_suave = np.gradient(f.desenfocar_gaussiano(gris, 2.0)[y_ini:y_fin, x0])

fig, ejes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ejes[0].plot(filas, perfil); ejes[0].set_title("intensidad a lo largo de la columna")
ejes[1].plot(filas, deriv_cruda, alpha=.6, label="derivada sin suavizar")
ejes[1].plot(filas, deriv_suave, lw=2, label="derivada de la imagen suavizada (σ=2)")
ejes[1].legend(); ejes[1].set_xlabel("fila (y)"); plt.show()

**Preguntas**
13. ¿Cuántos picos tiene la derivada alrededor de la línea? ¿Qué signo tiene cada uno y por qué?
14. Compará las dos derivadas: ¿cuál te permitiría encontrar el borde con un umbral simple? Conectalo con el apunte 03, sección 3.

✍️ **Respuesta:**

---
# Sesión 4 — Canny y Hough

## S4.a Canny paso a paso

In [ ]:
nms = f.supresion_no_maximos(mag, th)
bordes = f.canny(gris, sigma=1.4, bajo=0.1, alto=0.25)

import cv2
ref = cv2.Canny(gris.astype(np.uint8), 100, 200) > 0
mostrar(mag, nms > 0, bordes, ref, titulos=["magnitud", "después de NMS", "tu Canny", "cv2.Canny (comparación)"], ancho=5)
print("píxeles de borde: tuyo", bordes.sum(), "| OpenCV", ref.sum())

**Preguntas**
15. ¿Qué hizo la supresión de no máximos? ¿Qué agregó la histéresis después?
16. ¿En qué se parece y en qué difiere tu Canny del de OpenCV? (No tienen por qué dar igual: ¿por qué?)

✍️ **Respuesta:**

## S4.b Umbrales

In [ ]:
# TODO: mostrá tu Canny con (bajo, alto) = (0.05, 0.1), (0.1, 0.25), (0.2, 0.5) lado a lado

**Preguntas**
17. ¿Qué se pierde y qué se gana al subir los umbrales? ¿Qué combinación elegirías para encontrar las líneas de la cancha, y por qué?

✍️ **Respuesta:**

## S4.c Las líneas de la cancha: el pipeline completo

1. **Región de cancha:** máscara de pasto HSV → "cerrarla" (para tapar las líneas y huecos chicos) → **erosionarla** (para alejarse del borde con la tribuna).
2. **Cal:** dentro de la región, píxeles blancos (V alto, S bajo).
3. **Hough** sobre la cal → picos.
4. **Refinar** cada recta con cuadrados mínimos totales (TP1) usando los píxeles de cal cercanos.
5. **Intersecciones** (TP1).

In [ ]:
def dilatar(m, r):      # morfología binaria mínima, con ventanas (herramienta auxiliar, no es del TP)
    p = np.pad(m, r)
    return sliding_window_view(p, (2 * r + 1, 2 * r + 1)).any(axis=(-2, -1))

def erosionar(m, r):
    p = np.pad(m, r, constant_values=False)
    return sliding_window_view(p, (2 * r + 1, 2 * r + 1)).all(axis=(-2, -1))

pasto = f.mascara_pasto_hsv(img)
region = erosionar(dilatar(pasto, 6), 6)          # "cierre": tapa las líneas
region = erosionar(region, 15)                    # alejarse del borde con la tribuna
H_, S_, V_ = hsv[..., 0], hsv[..., 1], hsv[..., 2]
cal = region & (V_ > 0.75) & (S_ < 0.25)
mostrar(img, region, cal, titulos=["frame", "región de cancha", "candidatos a cal"], ancho=6)

In [ ]:
acc, thetas, rhos = f.hough_rectas(cal)
picos = f.picos_hough(acc, thetas, rhos, n=6, vecindad=(15, 8))
for r, t, v in picos:
    print(f"ρ = {r:7.1f}  θ = {np.rad2deg(t):6.1f}°  votos = {v}")

plt.figure(figsize=(12, 5))
plt.imshow(np.log1p(acc), aspect="auto", extent=[0, 180, rhos[-1], rhos[0]], cmap="magma")
plt.scatter([np.rad2deg(t) for _, t, _ in picos], [r for r, _, _ in picos], c="cyan", s=40)
plt.xlabel("θ (grados)"); plt.ylabel("ρ (px)"); plt.title("acumulador de Hough (log)"); plt.show()

In [ ]:
def dibujar_recta_abc(r, color, lw=1.5):
    a, b, c = r
    if abs(b) > abs(a):
        x = np.array([0, W]); y = -(a * x + c) / b
    else:
        y = np.array([0, H]); x = -(b * y + c) / a
    plt.plot(x, y, color=color, lw=lw)

ys, xs = np.nonzero(cal)
P_cal = np.column_stack([xs, ys]).astype(float)

rectas_hough = [f.recta_desde_hough(r, t) for r, t, _ in picos]
rectas_finas = []
# TODO: para cada recta de Hough:
#   1. quedate con los puntos de P_cal a menos de ~3 px (g.distancia_a_recta)
#   2. si hay suficientes (p. ej. > 50), ajustalos con g.ajustar_recta_total y agregala a rectas_finas

plt.figure(figsize=(14, 8)); plt.imshow(img)
for r in rectas_hough: dibujar_recta_abc(r, "yellow", 1)
for r in rectas_finas: dibujar_recta_abc(r, "cyan", 2)
plt.xlim(0, W); plt.ylim(H, 0); plt.title("Hough (amarillo) y refinadas (cian)"); plt.show()

In [ ]:
# TODO: calculá las intersecciones entre todas las rectas refinadas (g.interseccion),
# quedate con las que caen DENTRO de la imagen y dibujalas sobre el frame.

**Preguntas**
18. ¿Cuántas líneas reales de la cancha encontraste, y cuántas falsas? ¿De dónde salen las falsas?
19. ¿Te aparecieron "dos rectas por línea"? ¿Cómo lo resolviste?
20. ¿Cuántas intersecciones **útiles** (esquinas del área, cruces con la línea de medio) encontraste? ¿Alcanzarían las 4 que necesita una homografía (U4)? Si no: ¿qué te dice eso sobre automatizar la calibración con esta cámara?

✍️ **Respuesta:**

## Conclusiones

En 5–10 líneas: lo más importante de U2, qué del pipeline de la S4.c funcionaría en un partido entero y qué no, y qué te queda flojo.

✍️ **Respuesta:**